In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE
from sklearn.manifold import Isomap
import umap
from sklearn.manifold import trustworthiness
from sklearn.model_selection import GridSearchCV, ParameterGrid

In [12]:
data = pd.read_pickle('data_mod.pkl')
data.head()

,k1,k2,k3,k4,k5,k6,k7,k9,k10,k11,k12,k13,k14,k15,k16,k17,k18,k19,k20
0,0.403809,0.503518,0.548820,0.328302,0.330954,0.475458,0.833823,0.529053,0.330879,0.414133,0.451272,0.309298,0.359411,0.340312,0.459953,1.000000,0.554562,0.497247,0.479035
1,0.591695,0.653597,0.632007,0.763404,0.169329,0.543158,0.421875,0.903975,0.333300,0.515462,0.471764,0.330484,0.592674,0.566433,0.490934,1.000000,0.678800,0.510526,0.458055
2,0.300968,0.462057,0.341906,0.000000,0.379270,0.893999,1.000000,0.511229,0.284299,0.173041,0.502503,0.083725,0.220906,0.402367,1.000000,0.558105,0.000000,0.371303,0.399927
3,1.000000,1.000000,1.000000,0.902599,0.126387,0.311175,0.433346,0.999822,0.607228,0.489965,0.564472,1.000000,1.000000,0.942774,0.418881,0.546973,0.854628,1.000000,1.000000
4,0.555295,0.464936,0.620328,0.709942,0.235475,1.000000,0.927524,0.752390,0.374004,0.243528,0.407338,0.231825,0.403838,0.317202,0.420597,0.496557,0.516877,0.414231,0.426759


In [13]:
data = StandardScaler().fit_transform(data)

In [3]:
def scorer_func(estimator, data):
    return trustworthiness(data, estimator.transform(data), n_neighbors = 15)

In [24]:
kpca = KernelPCA(n_components = 7)
param_grid_kpca = [
    {
        'kernel': ['rbf'],
        'gamma': [0.00001, 0.00005, 0.0001, 0.0005, 0.001]
    },
    {
        'kernel': ['poly'],
        'gamma': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],
        'degree': [2, 3, 4]
    },
    {
        'kernel': ['sigmoid'],
        'gamma': [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
    }
]

grid_search_kernelpca = GridSearchCV(estimator = kpca, param_grid = param_grid_kpca, scoring = scorer_func, cv = 5)

In [23]:
grid_search_kernelpca.fit(data)
kpca_cv_df = pd.DataFrame(grid_search_kernelpca.cv_results_).sort_values(by = 'mean_test_score', ascending = False)
print(grid_search_kernelpca.best_params_, '\n', grid_search_kernelpca.best_score_)
kpca_cv_df[['param_gamma', 'param_kernel', 'mean_test_score','std_test_score', 'rank_test_score']]

{'gamma': 0.01, 'kernel': 'sigmoid'} 
 0.982893193923094


,param_gamma,param_kernel,mean_test_score,std_test_score,rank_test_score
29,0.01000,sigmoid,0.982893,0.001282,1
28,0.00500,sigmoid,0.982824,0.001312,2
6,0.00050,poly,0.982623,0.001205,3
20,0.00050,poly,0.982619,0.001256,4
5,0.00010,poly,0.982610,0.001217,5
0,0.00001,rbf,0.982606,0.001220,6
12,0.00010,poly,0.982599,0.001220,7
27,0.00100,sigmoid,0.982598,0.001271,8
2,0.00010,rbf,0.982598,0.001174,9
19,0.00010,poly,0.982595,0.001209,10


In [16]:
res = []

param_grid = {
    'n_components': [2],
    'perplexity': [5, 15, 30, 50],
    'learning_rate': [50.0, 100.0, 200.0, 500.0, 1000.0],
}

for params in ParameterGrid(param_grid):
    tsne = TSNE(**params, random_state = 42)
    
    data_emb = tsne.fit_transform(data)
    score = trustworthiness(data, data_emb, n_neighbors = 15)
    
    res_entry = {f'param_{k}': v for k, v in params.items()}
    res_entry['mean_test_score'] = score
    res.append(res_entry)

tsne_cv_df = pd.DataFrame(res).sort_values(by = 'mean_test_score', ascending = False)

best_it = tsne_cv_df.iloc[0]
print({k.replace('param_', ''): v for k, v in best_it.items() if k.startswith('param_')})
print(best_it['mean_test_score'])
display(tsne_cv_df)

{'learning_rate': 50.0, 'n_components': 2.0, 'perplexity': 50.0}
0.9698451427386414


,param_learning_rate,param_n_components,param_perplexity,mean_test_score
3,50.0,2,50,0.969845
7,100.0,2,50,0.968939
6,100.0,2,30,0.966946
2,50.0,2,30,0.966938
1,50.0,2,15,0.966904
18,1000.0,2,30,0.966828
10,200.0,2,30,0.966805
11,200.0,2,50,0.966044
9,200.0,2,15,0.965871
13,500.0,2,15,0.965837


In [28]:
isomap = Isomap()
param_grid_isomap = [
    {
        'n_neighbors': [5, 10, 15, 30, 50],
        'radius': [None],
        'n_components': [2, 7],
        'metric': ['euclidean', 'manhattan']
    },
    {
        'n_neighbors': [None],
        'radius': [1.0, 2.0, 3.0],
        'n_components': [2, 7],
        'metric': ['euclidean', 'manhattan']
    }
]

grid_search_isomap = GridSearchCV(estimator = isomap, param_grid = param_grid_isomap, scoring = scorer_func, cv = 5)

In [ ]:
grid_search_isomap.fit(data)
isomap_cv_df = pd.DataFrame(grid_search_isomap.cv_results_).sort_values(by = 'mean_test_score', ascending = False)
print(grid_search_isomap.best_params_, '\n', grid_search_isomap.best_score_)
isomap_cv_df[['param_n_neighbors', 'param_radius', 'n_components', 'param_metric', 'mean_test_score','std_test_score', 'rank_test_score']]

In [ ]:
umap = Umap()

param_grid_umap = {
    'n_neighbors': [5, 15, 30, 50],
    'min_dist': [0.0, 0.1, 0.3, 0.5],
    'n_components': [2, 7],
    'metric': ['euclidean', 'manhattan']
}

grid_search_umap = GridSearchCV(estimator = umap, param_grid = param_grid_umap, scoring = scorer_func, cv = 5)